Perform text preprocessing by applying advanced NLP techniques such as Named Entity Recognition (NER) filtering, part-of-speech tagging for context-aware stopword removal, and domain-specific text cleaning beyond standard tokenization, stemming, and lemma

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [2]:
df = pd.read_csv("task21_final_dataset.csv")
df.shape

(23483, 46)

In [3]:
df.head(5)

,product_id,category,subcategory1,subcategory2,location,channel,customer_age,review_title,review_text,rating,...,location_Mumbai,channel_Mobile,channel_Web,vader_sentiment_label_Negative,vader_sentiment_label_Neutral,vader_sentiment_label_Positive,age_group_Age Group 1,age_group_Age Group 2,age_group_Age Group 3,age_group_Age Group 4
0,767,Initmates,Intimate,Intimates,Mumbai,Mobile,33,Not Provided,Absolutely wonderful - silky and sexy and comfortable,4,...,1,1,0,0,0,1,0,1,0,0
1,1080,General,Dresses,Dresses,Bangalore,Mobile,34,Not Provided,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite.",5,...,0,1,0,0,0,1,0,1,0,0
2,1077,General,Dresses,Dresses,Gurgaon,Mobile,60,Some major design flaws,"I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c",3,...,0,1,0,0,0,1,0,0,0,1
3,1049,General Petite,Bottoms,Pants,Chennai,Web,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!",5,...,0,0,1,0,0,1,0,0,1,0
4,847,General,Tops,Blouses,Bangalore,Web,47,Flattering shirt,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!,5,...,0,0,1,0,0,1,0,0,1,0


In [4]:
df.columns.to_list()

['product_id',
 'category',
 'subcategory1',
 'subcategory2',
 'location',
 'channel',
 'customer_age',
 'review_title',
 'review_text',
 'rating',
 'recommend_flag',
 'vader_negative',
 'vader_neutral',
 'vader_positive',
 'vader_compound',
 'vader_sentiment_score',
 'vader_sentiment_label',
 'age_cluster_raw',
 'age_group',
 'sentiment_target',
 'text_clean',
 'text_stemmed',
 'text_lemmatized',
 'review_title_binned',
 'category_General',
 'category_General Petite',
 'category_Initmates',
 'subcategory1_Bottoms',
 'subcategory1_Dresses',
 'subcategory1_Intimate',
 'subcategory1_Jackets',
 'subcategory1_Tops',
 'subcategory1_Trend',
 'location_Bangalore',
 'location_Chennai',
 'location_Gurgaon',
 'location_Mumbai',
 'channel_Mobile',
 'channel_Web',
 'vader_sentiment_label_Negative',
 'vader_sentiment_label_Neutral',
 'vader_sentiment_label_Positive',
 'age_group_Age Group 1',
 'age_group_Age Group 2',
 'age_group_Age Group 3',
 'age_group_Age Group 4']

In [5]:
for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        if {"review_title", "review_text", "text_clean"}.issubset(obj.columns):
            print("DataFrame:", name)
            print("Shape:", obj.shape)
            print("\nSample reviews:")
            print(obj[["review_title", "review_text", "text_clean"]].head(5))
            print("-" * 60)

DataFrame: __
Shape: (5, 46)

Sample reviews:
              review_title  \
0             Not Provided   
1             Not Provided   
2  Some major design flaws   
3         My favorite buy!   
4         Flattering shirt   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            review_text  \
0                                                                                                                                                                                                                                                                         

In [7]:
%pip install spacy

   ---------------------------------------- 0.0/15.2 MB ? eta -:--:--
    --------------------------------------- 0.3/15.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/15.2 MB 1.8 MB/s eta 0:00:09
   -- ------------------------------------- 1.0/15.2 MB 1.9 MB/s eta 0:00:08
   --- ------------------------------------ 1.3/15.2 MB 1.9 MB/s eta 0:00:08
   ---- ----------------------------------- 1.8/15.2 MB 1.9 MB/s eta 0:00:08
   ------ --------------------------------- 2.4/15.2 MB 2.0 MB/s eta 0:00:07
   ------- -------------------------------- 2.9/15.2 MB 2.1 MB/s eta 0:00:06
   -------- ------------------------------- 3.4/15.2 MB 2.1 MB/s eta 0:00:06
   ---------- ----------------------------- 3.9/15.2 MB 2.2 MB/s eta 0:00:06
   ----------- ---------------------------- 4.5/15.2 MB 2.3 MB/s eta 0:00:05
   ------------- -------------------------- 5.2/15.2 MB 2.3 MB/s eta 0:00:05
   --------------- ------------------------ 5.8/15.2 MB 2.3 MB/s eta 0:00:05
   ----------


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     --- ------------------------------------ 1.0/12.8 MB 2.6 MB/s eta 0:00:05
     ---- ----------------------------------- 1.6/12.8 MB 2.6 MB/s eta 0:00:05
     ------ --------------------------------- 2.1/12.8 MB 2.8 MB/s eta 0:00:04
     --------- ------------------------------ 2.9/12.8 MB 2.9 MB/s eta 0:00:04
     ----------- ---------------------------- 3.7/12.8 MB 3.0 MB/s eta 0:00:04
     --------------- ------------------------ 5.0/12.8 MB 3.5 MB/s eta 0:00:03
     ------------------ --------------------- 5.8/12.8 MB 3.6 MB/s eta 0:00:02
     --------------------- ------------------ 6.8/12.8 MB 3.7 MB/s eta 0:00:02
     ------------------------ --------------- 7.9/12.8 MB 3.9 MB/s eta 0:00:02
     --------------------------- ------------ 8.9/12.8 MB 4.0 MB/s eta 0:


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import spacy

nlp = spacy.load("en_core_web_sm")

print("spaCy Version:", spacy.__version__)
print("Pipeline Components:", nlp.pipe_names)
print("English NLP Model Loaded Successfully!")

spaCy Version: 3.8.18
Pipeline Components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
English NLP Model Loaded Successfully!


Perform Named Entity Recognition (NER)

In [10]:
sample_reviews = df['review_text'].dropna().head(10)

for i, review in enumerate(sample_reviews, 1):

    doc = nlp(review)
    print(f"\nREVIEW {i}")
    print("-" * 60)

    print("Review:", review[:250])

    entities = [(ent.text, ent.label_)for ent in doc.ents]

    print("Named Entities:", entities)


REVIEW 1
------------------------------------------------------------
Review: Absolutely wonderful - silky and sexy and comfortable
Named Entities: []

REVIEW 2
------------------------------------------------------------
Review: Love this dress!  it's sooo pretty.  i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite.  i bought a petite and am 5'8".  i love the length on me- hits just a little below the knee.  would defi
Named Entities: [("5'8", 'CARDINAL')]

REVIEW 3
------------------------------------------------------------
Review: I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, wh
Named Entities: [('half', 'CARDINAL'), ('half', 'CARDINAL')]

REVIEW 4
------------------------------------------------------------
Review: I love, lo

In [11]:
from tqdm.notebook import tqdm

In [13]:
# Entity labels to filter
remove_labels = {
    "PERSON",
    "ORG",
    "GPE",
    "LOC",
    "FAC",
    "CARDINAL",
    "QUANTITY",
    "MONEY",
    "PERCENT",
    "DATE"
}

def remove_irrelevant_entities(text):
    if pd.isna(text):
        return ""
    doc = nlp(str(text))
    # Store entity character positions
    remove_spans = [
        (ent.start_char, ent.end_char)
        for ent in doc.ents
        if ent.label_ in remove_labels
    ]

    result = str(text)

    for start, end in reversed(remove_spans):
        result = result[:start] + " " + result[end:]
    return result

# Apply NER filtering without tqdm
df["text_ner_filtered"] = df["review_text"].apply(remove_irrelevant_entities)

print("NER FILTERING COMPLETED")
print("-----------------------")
print("Original Shape:", df.shape)
print("New Column:", "text_ner_filtered")

print("\nBefore and After Comparison:")

display(
    df[["review_text", "text_ner_filtered"]].head(10)
)

NER FILTERING COMPLETED
-----------------------
Original Shape: (23483, 47)
New Column: text_ner_filtered

Before and After Comparison:


,review_text,text_ner_filtered
0,Absolutely wonderful - silky and sexy and comfortable,Absolutely wonderful - silky and sexy and comfortable
1,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite.","Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am "". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite."
2,"I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c","I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top was comfortable and fit nicely, but the bottom had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c"
3,"I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!","I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!"
4,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!
5,"I love tracy reese dresses, but this one is not for the very petite. i am just under 5 feet tall and usually wear a 0p in this brand. this dress was very pretty out of the package but its a lot of dress. the skirt is long and very full so it overwhelmed my small frame. not a stranger to alterations, shortening and narrowing the skirt would take away from the embellishment of the garment. i love the color and the idea of the style but it just did not work on me. i returned this dress.","I love tracy reese dresses, but this one is not for the very petite. i am tall and usually wear a in this brand. this dress was very pretty out of the package but its a lot of dress. the skirt is long and very full so it overwhelmed my small frame. not a stranger to alterations, shortening and narrowing the skirt would take away from the embellishment of the garment. i love the color and the idea of the style but it just did not work on me. i returned this dress."
6,"I aded this in my basket at hte last mintue to see what it would look like in person. (store pick up). i went with teh darkler color only because i am so pale :-) hte color is really gorgeous, and turns out it mathced everythiing i was trying on with it prefectly. it is a little baggy on me and hte xs is hte msallet size (bummer, no petite). i decided to jkeep it though, because as i said, it matvehd everything. my ejans, pants, and the 3 skirts i waas trying on (of which i ]kept all ) oops.","I aded this in my basket at last mintue to see what it would look like in person. (store pick up). i went with teh darkler color only because i am so

In [14]:
# Keep numerical and measurement information
remove_labels = {
    "PERSON", "ORG", "GPE", "LOC", "FAC"
}

def remove_irrelevant_entities(text):
    if pd.isna(text):
        return ""

    doc = nlp(str(text))
    result = str(text)

    remove_spans = [
        (ent.start_char, ent.end_char)
        for ent in doc.ents
        if ent.label_ in remove_labels
    ]

    for start, end in reversed(remove_spans):
        result = result[:start] + " " + result[end:]

    return result

df["text_ner_filtered"] = df["review_text"].apply(
    remove_irrelevant_entities
)

display(df[["review_text", "text_ner_filtered"]].head(10))

,review_text,text_ner_filtered
0,Absolutely wonderful - silky and sexy and comfortable,Absolutely wonderful - silky and sexy and comfortable
1,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite.","Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite."
2,"I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c","I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c"
3,"I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!","I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!"
4,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!
5,"I love tracy reese dresses, but this one is not for the very petite. i am just under 5 feet tall and usually wear a 0p in this brand. this dress was very pretty out of the package but its a lot of dress. the skirt is long and very full so it overwhelmed my small frame. not a stranger to alterations, shortening and narrowing the skirt would take away from the embellishment of the garment. i love the color and the idea of the style but it just did not work on me. i returned this dress.","I love tracy reese dresses, but this one is not for the very petite. i am just under 5 feet tall and usually wear a 0p in this brand. this dress was very pretty out of the package but its a lot of dress. the skirt is long and very full so it overwhelmed my small frame. not a stranger to alterations, shortening and narrowing the skirt would take away from the embellishment of the garment. i love the color and the idea of the style but it just did not work on me. i returned this dress."
6,"I aded this in my basket at hte last mintue to see what it would look like in person. (store pick up). i went with teh darkler color only because i am so pale :-) hte color is really gorgeous, and turns out it mathced everythiing i was trying on with it prefectly. it is a little baggy on me and hte xs is hte msallet size (bummer, no petite). i decided to jkeep it though, because as i said, it matvehd everything. my ejans, pants, and the 3 skirts i waas trying on (of which i ]kept all ) oops.","I aded this in my basket at last mintue to see what it would look like in person. (store pick up). i went with teh 

In [15]:
# POS Tagging - Sample Inspection

sample_reviews = df["text_ner_filtered"].dropna().head(5)

for i, review in enumerate(sample_reviews, 1):
    doc = nlp(review)

    print(f"\nREVIEW {i}")
    print("-" * 60)

    for token in doc:
        if not token.is_punct and not token.is_space:
            print(f"{token.text:20} {token.pos_:10} {token.tag_}")


REVIEW 1
------------------------------------------------------------
Absolutely           ADV        RB
wonderful            ADJ        JJ
silky                NOUN       NN
and                  CCONJ      CC
sexy                 ADJ        JJ
and                  CCONJ      CC
comfortable          ADJ        JJ

REVIEW 2
------------------------------------------------------------
Love                 VERB       VB
this                 DET        DT
dress                NOUN       NN
it                   PRON       PRP
's                   AUX        VBZ
sooo                 NOUN       NN
pretty               ADV        RB
i                    PRON       PRP
happened             VERB       VBD
to                   PART       TO
find                 VERB       VB
it                   PRON       PRP
in                   ADP        IN
a                    DET        DT
store                NOUN       NN
and                  CCONJ      CC
i                    PRON       PRP
'm          

Context-aware stopword removal

In [18]:
preserve_words = {'not', 'no', 'never', 'nor'}
remove_pos = {"DET", "PRON", "AUX", "ADP","CCONJ", "SCONJ", "PART"}

sample_reviews = df['text_ner_filtered'].dropna().head(2000)

for i, review in enumerate(sample_reviews, 1):
    doc = nlp(review)

    filtered_tokens = [
        token.text
        for token in doc if(token.pos_ not in remove_pos and not token.is_punct and not token.is_space) or token.text.lower() in preserve_words
    ]

    print(f"\nREVIEW {i}")
    print("-" * 60)
    print("Original:", review)
    print("Filtered:", " ".join(filtered_tokens))


REVIEW 1
------------------------------------------------------------
Original: Absolutely wonderful - silky and sexy and comfortable
Filtered: Absolutely wonderful silky sexy comfortable

REVIEW 2
------------------------------------------------------------
Original: Love this dress!  it's sooo pretty.  i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite.  i bought a petite and am 5'8".  i love the length on me- hits just a little below the knee.  would definitely be a true midi on someone who is truly petite.
Filtered: Love dress sooo pretty happened find store glad did bc never ordered online bc petite bought petite 5'8 love length me- hits just little knee definitely true midi truly petite

REVIEW 3
------------------------------------------------------------
Original: I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageou

Apply context-aware stopword removal to full dataset

In [20]:
# Context-aware Stopword Removal

preserve_words = {"not", "no", "never", "nor"}

remove_pos = {
    "DET", "PRON", "AUX", "ADP",
    "CCONJ", "SCONJ", "PART"
}

def context_aware_cleaning(text):

    if pd.isna(text):
        return ""

    doc = nlp(str(text))

    filtered_tokens = [
        token.text
        for token in doc
        if (
            token.pos_ not in remove_pos
            and not token.is_punct
            and not token.is_space
        ) or token.text.lower() in preserve_words
    ]

    return " ".join(filtered_tokens)


# Create the new column from the NER-filtered text
df["text_pos_filtered"] = df["text_ner_filtered"].apply(
    context_aware_cleaning
)

print("CONTEXT-AWARE STOPWORD REMOVAL COMPLETED")
print("----------------------------------------")
print("Dataset Shape:", df.shape)
print("New Column:", "text_pos_filtered")
print("Total Processed Reviews:", len(df))

display(
    df[["text_ner_filtered", "text_pos_filtered"]].head(10)
)

CONTEXT-AWARE STOPWORD REMOVAL COMPLETED
----------------------------------------
Dataset Shape: (23483, 48)
New Column: text_pos_filtered
Total Processed Reviews: 23483


,text_ner_filtered,text_pos_filtered
0,Absolutely wonderful - silky and sexy and comfortable,Absolutely wonderful silky sexy comfortable
1,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite.",Love dress sooo pretty happened find store glad did bc never ordered online bc petite bought petite 5'8 love length me- hits just little knee definitely true midi truly petite
2,"I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c",had such high hopes dress really wanted work initially ordered petite small usual size found outrageously small so small fact not zip reordered petite medium just ok overall top half comfortable fit nicely bottom half had very tight layer several somewhat cheap net layers imo major design flaw net layer sewn directly zipper c
3,"I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!",love love love jumpsuit fun flirty fabulous time wear get great compliments
4,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!,shirt very flattering due adjustable front tie perfect length wear leggings sleeveless so pairs well cardigan love shirt
5,"I love tracy reese dresses, but this one is not for the very petite. i am just under 5 feet tall and usually wear a 0p in this brand. this dress was very pretty out of the package but its a lot of dress. the skirt is long and very full so it overwhelmed my small frame. not a stranger to alterations, shortening and narrowing the skirt would take away from the embellishment of the garment. i love the color and the idea of the style but it just did not work on me. i returned this dress.",love tracy reese dresses one not very petite just 5 feet tall usually wear 0p brand dress very pretty package lot dress skirt long very full so overwhelmed small frame not stranger alterations shortening narrowing skirt take away embellishment garment love color idea style just not work returned dress
6,"I aded this in my basket at last mintue to see what it would look like in person. (store pick up). i went with teh darkler color only because i am so pale :-) hte color is really gorgeous, and turns out it mathced everythiing i was trying on with it prefectly. it is a little baggy on me and is msallet size (bummer, no petite). i decided to jkeep it though, because as i said, it matvehd everything. my ejans, pants, and the 3 skirts i waas trying on (of which i ]kept all ) oops.",aded basket last mintue see look person store pick went teh darkler color only so pale hte color really gorgeous turns mathced everythiing trying prefectly little baggy msallet size bummer no petite decided jkeep though said matvehd ejans pants 3 skirts waas trying kept oops
7,"I ordered this in carbon for store pick up, and had a ton of stuff (as always) to try on and used this top to pair (skirts and pants). everything went with it. the color is really nice charcoal with shimmer, and went well with pencil skirts, flare pants, etc. my only compaint is it is a bit big, sleeves are long and it doesn't go in petite. also a bit loose for me, but no xxs... so i kept it and wil ldecide later since the light color is already sold out in smalle

Fashion-domain-specific text cleaning

In [21]:
from collections import Counter
import re

In [22]:
all_text = " ".join(df['text_pos_filtered'].dropna()).lower()

words = re.findall(r"\b[a-z0-9]+\b", all_text)

word_counts = Counter(words)

print("TOP 30 MOST COMMON WORDS")
print("-------------------------")

for word, count in word_counts.most_common(30):
    print(f"{word:20} {count}")

TOP 30 MOST COMMON WORDS
-------------------------
not                  10822
dress                10559
so                   9542
love                 8945
size                 8744
very                 8213
top                  7405
fit                  7316
wear                 6435
great                6111
too                  6054
just                 5608
5                    5558
have                 4832
fabric               4780
small                4721
color                4593
look                 3995
more                 3937
really               3923
ordered              3850
little               3773
perfect              3773
one                  3563
flattering           3516
well                 3388
soft                 3339
back                 3141
comfortable          3057
cute                 3039


Fashion-domain-specific cleaning

In [24]:
abbreviation_map = {
    "bc": "because",
    "imo": "in my opinion",
    "w/": "with",
    "w/o": "without",
    "xxs": "xxs",
}

def fashion_domain_cleaning(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()

    # Normalize common abbreviations
    for short, full in abbreviation_map.items():
        text = re.sub(rf"\b{re.escape(short)}\b", full, text)
        
    # Normalize repeated characters (soooo -> soo)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    # Remove unwanted symbols but preserve numbers, apostrophes, hyphens and useful measurement characters
    text = re.sub(r"[^a-z0-9\s.'%/-]", " ", text)

     # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


df['text_domain_cleaned'] = df['text_pos_filtered'].apply(fashion_domain_cleaning)

print("FASHION DOMAIN CLEANING COMPLETED")
print("---------------------------------")
print("Dataset Shape:", df.shape)
print("New Column: text_domain_cleaned")

display(df[["text_pos_filtered", "text_domain_cleaned"]].head(10))

FASHION DOMAIN CLEANING COMPLETED
---------------------------------
Dataset Shape: (23483, 49)
New Column: text_domain_cleaned


,text_pos_filtered,text_domain_cleaned
0,Absolutely wonderful silky sexy comfortable,absolutely wonderful silky sexy comfortable
1,Love dress sooo pretty happened find store glad did bc never ordered online bc petite bought petite 5'8 love length me- hits just little knee definitely true midi truly petite,love dress soo pretty happened find store glad did because never ordered online because petite bought petite 5'8 love length me- hits just little knee definitely true midi truly petite
2,had such high hopes dress really wanted work initially ordered petite small usual size found outrageously small so small fact not zip reordered petite medium just ok overall top half comfortable fit nicely bottom half had very tight layer several somewhat cheap net layers imo major design flaw net layer sewn directly zipper c,had such high hopes dress really wanted work initially ordered petite small usual size found outrageously small so small fact not zip reordered petite medium just ok overall top half comfortable fit nicely bottom half had very tight layer several somewhat cheap net layers in my opinion major design flaw net layer sewn directly zipper c
3,love love love jumpsuit fun flirty fabulous time wear get great compliments,love love love jumpsuit fun flirty fabulous time wear get great compliments
4,shirt very flattering due adjustable front tie perfect length wear leggings sleeveless so pairs well cardigan love shirt,shirt very flattering due adjustable front tie perfect length wear leggings sleeveless so pairs well cardigan love shirt
5,love tracy reese dresses one not very petite just 5 feet tall usually wear 0p brand dress very pretty package lot dress skirt long very full so overwhelmed small frame not stranger alterations shortening narrowing skirt take away embellishment garment love color idea style just not work returned dress,love tracy reese dresses one not very petite just 5 feet tall usually wear 0p brand dress very pretty package lot dress skirt long very full so overwhelmed small frame not stranger alterations shortening narrowing skirt take away embellishment garment love color idea style just not work returned dress
6,aded basket last mintue see look person store pick went teh darkler color only so pale hte color really gorgeous turns mathced everythiing trying prefectly little baggy msallet size bummer no petite decided jkeep though said matvehd ejans pants 3 skirts waas trying kept oops,aded basket last mintue see look person store pick went teh darkler color only so pale hte color really gorgeous turns mathced everythiing trying prefectly little baggy msallet size bummer no petite decided jkeep though said matvehd ejans pants 3 skirts waas trying kept oops
7,ordered carbon store pick had ton stuff always try used top pair skirts pants went color really nice charcoal shimmer went well pencil skirts flare pants etc only compaint bit big sleeves long go petite also bit loose no xxs so kept wil ldecide later light color already sold smallest size,ordered carbon store pick had ton stuff always try used top pair skirts pants went color really nice charcoal shimmer went well pencil skirts flare pants etc only compaint bit big sleeves long go petite also bit loose no xxs so kept wil ldecide later light color already sold smallest size
8,love dress usually get xs runs little snug bust so ordered size very flattering feminine usual retailer flair style,love dress usually get xs runs little snug bust so ordered size very flattering feminine usual retailer flair style
9,"5""5 125 lbs ordered s petite make sure length too long typically wear xs regular retailer dresses less busty 34b cup smaller s petite fit perfectly snug not tight love dress party down work love tulle longer then fabric underneath",5 5 125 lbs ordered s petite make sure length too long typically wear xs regular retailer dresses less busty 34b cup smaller s petite fit perfectly snug not tight love dress party down work love tulle longer then fabric undernea

In [25]:
# Final NLP preprocessing validation

print("FINAL NLP PREPROCESSING VALIDATION")
print("=" * 45)

print("Dataset Shape:", df.shape)

columns_to_check = [
    "text_ner_filtered",
    "text_pos_filtered",
    "text_domain_cleaned"
]

print("\nMissing Values:")
print(df[columns_to_check].isnull().sum())

print("\nEmpty Text Counts:")
for col in columns_to_check:
    empty_count = (df[col].str.strip() == "").sum()
    print(f"{col}: {empty_count}")

print("\nSample Comparison:")
display(
    df[
        [
            "review_text",
            "text_ner_filtered",
            "text_pos_filtered",
            "text_domain_cleaned"
        ]
    ].head(10)
)

FINAL NLP PREPROCESSING VALIDATION
Dataset Shape: (23483, 49)

Missing Values:
text_ner_filtered      0
text_pos_filtered      0
text_domain_cleaned    0
dtype: int64

Empty Text Counts:
text_ner_filtered: 0
text_pos_filtered: 0
text_domain_cleaned: 0

Sample Comparison:


,review_text,text_ner_filtered,text_pos_filtered,text_domain_cleaned
0,Absolutely wonderful - silky and sexy and comfortable,Absolutely wonderful - silky and sexy and comfortable,Absolutely wonderful silky sexy comfortable,absolutely wonderful silky sexy comfortable
1,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite.","Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite.",Love dress sooo pretty happened find store glad did bc never ordered online bc petite bought petite 5'8 love length me- hits just little knee definitely true midi truly petite,love dress soo pretty happened find store glad did because never ordered online because petite bought petite 5'8 love length me- hits just little knee definitely true midi truly petite
2,"I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c","I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c",had such high hopes dress really wanted work initially ordered petite small usual size found outrageously small so small fact not zip reordered petite medium just ok overall top half comfortable fit nicely bottom half had very tight layer several somewhat cheap net layers imo major design flaw net layer sewn directly zipper c,had such high hopes dress really wanted work initially ordered petite small usual size found outrageously small so small fact not zip reordered petite medium just ok overall top half comfortable fit nicely bottom half had very tight layer several somewhat cheap net layers in my opinion major design flaw net layer sewn directly zipper c
3,"I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!","I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!",love love love jumpsuit fun flirty fabulous time wear get great compliments,love love love jumpsuit fun flirty fabulous time wear get great compliments
4,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!,shirt very flattering due adjustable front tie perfect length wear leggings sleeveless so pairs well cardigan love shirt,shirt very flattering due adjustable front tie perfect length wear leggings sleeveless so pairs well cardigan love shirt
5,"I love tracy reese dresses, but this one is not for

### Conclusion – Advanced NLP Text Preprocessing

Advanced NLP techniques were applied to the Women's Clothing Reviews dataset using spaCy and Python.

1. **Named Entity Recognition (NER):** Identified named entities and filtered selected irrelevant entity categories while preserving important fashion-related information.
2. **Part-of-Speech (POS) Tagging:** Identified grammatical categories such as nouns, verbs, adjectives, and adverbs.
3. **Context-Aware Stopword Removal:** Removed selected grammatical stopwords while preserving important negation words such as "not", "no", and "never" to maintain sentiment meaning.
4. **Domain-Specific Text Cleaning:** Normalized common abbreviations, repeated characters, letter casing, unwanted symbols, and extra spaces while retaining important fashion terminology.

**Final Result:** The dataset contains 23,483 reviews and 49 columns. The three newly generated preprocessing columns contain zero missing values and zero empty strings.

**Limitations:** Some NER classifications may remove relevant brand names, POS tagging may occasionally misclassify words, and domain-specific cleaning does not correct all spelling errors or preserve every measurement format.

Overall, the task successfully demonstrates the application of advanced NLP preprocessing techniques to fashion product reviews.
